# Masked Autoencoders (MAE) for MRI Images

**Vision Transformers with Self-Supervised Learning**

This notebook teaches **Masked Autoencoders (MAE)** — a self-supervised method for training Vision Transformers without labels. We use HuggingFace tools throughout and focus on MRI images only for fast, focused learning.

**What you will learn:**
1. Load a **small subset of MRI images** from ROCOv2 (fast data loading)
2. Understand how **MAE masks ~75% of image patches** and reconstructs them
3. Use a **pretrained ViT-MAE** from HuggingFace (no from-scratch building)
4. **Fine-tune MAE** on MRI domain for better medical image representations
5. **Compare reconstructions** before vs after training

---

**Table of Contents**

| Section | Topic |
|---------|-------|
| 0 | Setup |
| 1 | Load MRI Images (Limited Sample) |
| 2 | How MAE Masking Works |
| 3 | Load Pretrained ViT-MAE |
| 4 | Reconstructions Before Training |
| 5 | Fine-Tune MAE on MRI |
| 6 | Reconstructions After Training |
| 7 | Before vs After Comparison |
| 8 | Summary |

> **Colab:** Use *Runtime → Change runtime type → T4 GPU* for faster training.

In [1]:
# ── Section 0: Setup ───────────────────────────────────────────────────────
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(["Pillow>=10.0,<11.0"])
pip_install(["-U", "transformers", "datasets", "accelerate", "torchvision", "huggingface_hub"])

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
import transformers
from transformers import (
    ViTImageProcessor,
    ViTMAEForPreTraining,
    Trainer,
    TrainingArguments,
)
import math, re

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})
USE_PRECOMPUTED = False

torch: 2.10.0+cu128
transformers: 5.3.0
Device: cuda


---

## Section 1: Load MRI Images Only (Limited Sample)

We load **ROCOv2** from HuggingFace but:
- **Filter to MRI only** — much smaller subset, faster loading
- **Limit to 80% train / 20% val** — fast training, enough to see learning
- No labels needed — MAE is self-supervised (reconstruct pixels)

In [2]:
# Load only a small slice of ROCOv2 for speed, then filter to MRI
print("Loading ROCOv2 (first 2000 examples only — keeps load time short)...")
try:
    raw = load_dataset("eltorio/ROCOv2-radiology", split="train[:2000]")
except Exception:
    raw = load_dataset("akahana/rocov2-full", split="train[:2000]")

caption_col = "caption" if "caption" in raw.column_names else raw.column_names[1]
image_col = "image" if "image" in raw.column_names else raw.column_names[0]

def is_mri(ex):
    t = ex[caption_col].lower()
    return any(k in t for k in ["mri", "magnetic resonance", "mr image", "t1", "t2", "flair", "dwi"])

mri_ds = raw.filter(is_mri)
if len(mri_ds) == 0:
    print("No MRI found in sample — using random subset for demo")
    mri_ds = raw.shuffle(seed=SEED).select(range(500))
else:
    print(f"MRI images found: {len(mri_ds)}")

# Safe 80/20 split — works for any dataset size
shuffled = mri_ds.shuffle(seed=SEED)
N_TRAIN = min(200, max(1, int(len(shuffled) * 0.8)))
N_VAL   = len(shuffled) - N_TRAIN
train_ds = shuffled.select(range(N_TRAIN))
val_ds   = shuffled.select(range(N_TRAIN, N_TRAIN + N_VAL))

print(f"Using {len(train_ds)} train, {len(val_ds)} val")
print(f"Columns: {mri_ds.column_names}")

Loading ROCOv2 (first 2000 examples only — keeps load time short)...


KeyboardInterrupt: 

In [ ]:
# Quick gallery of our MRI subset
def to_rgb(img):
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    return img.convert("RGB")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < len(train_ds):
        img = to_rgb(train_ds[i][image_col])
        ax.imshow(img)
    ax.axis("off")
plt.suptitle("Sample MRI Images from ROCOv2", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

---

## Section 2: How MAE Masking Works

**Masked Autoencoders (He et al., 2021)** train ViT by:

1. **Split** image into 16×16 patches (e.g., 14×14 = 196 patches for 224×224)
2. **Mask ~75%** of patches — replace them with a learnable [MASK] token (or simply omit)
3. **Encoder** sees only the **visible 25%** of patches
4. **Decoder** reconstructs the **masked 75%** from encoder outputs + mask tokens
5. **Loss** = MSE between reconstructed and original pixel values (on masked patches only)

Why 75%? High masking ratio forces the model to learn strong semantic representations — it must infer global structure from very little local evidence.

In [ ]:
# Visualize MAE masking on one MRI image
IMG_SIZE, PATCH_SIZE = 224, 16
N_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
MASK_RATIO = 0.75

np.random.seed(SEED)
img = to_rgb(train_ds[0][image_col]).resize((IMG_SIZE, IMG_SIZE))
arr = np.array(img)

# Randomly choose 75% of patches to mask
ids = np.random.permutation(N_PATCHES)
n_keep = int(N_PATCHES * (1 - MASK_RATIO))
ids_keep = ids[:n_keep]
ids_mask = ids[n_keep:]
mask = np.ones(N_PATCHES, dtype=bool)
mask[ids_keep] = False

# Build masked image (gray for masked patches)
masked_arr = arr.copy()
grid_h = IMG_SIZE // PATCH_SIZE
for idx in ids_mask:
    i, j = idx // grid_h, idx % grid_h
    y1, y2 = i * PATCH_SIZE, (i + 1) * PATCH_SIZE
    x1, x2 = j * PATCH_SIZE, (j + 1) * PATCH_SIZE
    masked_arr[y1:y2, x1:x2] = 128  # gray

# Build "visible only" (black for masked)
visible_arr = np.zeros_like(arr)
for idx in ids_keep:
    i, j = idx // grid_h, idx % grid_h
    y1, y2 = i * PATCH_SIZE, (i + 1) * PATCH_SIZE
    x1, x2 = j * PATCH_SIZE, (j + 1) * PATCH_SIZE
    visible_arr[y1:y2, x1:x2] = arr[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(arr); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(masked_arr); axes[1].set_title("Masked (gray = 75% removed)"); axes[1].axis("off")
axes[2].imshow(visible_arr); axes[2].set_title("Visible only (what encoder sees)"); axes[2].axis("off")
plt.suptitle(f"MAE Masking: {n_keep} visible / {N_PATCHES} total patches ({100*(1-MASK_RATIO):.0f}% visible)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("→ The encoder only processes the colored patches. The decoder must reconstruct the gray ones.")

---

## Section 3: Load Pretrained ViT-MAE

We use **`facebook/vit-mae-base`** from HuggingFace — a general-purpose ViT pretrained on ImageNet with MAE. It is the smallest official MAE on HuggingFace (~86M params) and trains quickly on our small MRI set.

- **ViTImageProcessor**: Resizes to 224×224, normalizes with ImageNet stats
- **ViTMAEForPreTraining**: Full model (encoder + decoder) for MAE training

In [ ]:
MODEL_ID = "facebook/vit-mae-base"

processor = ViTImageProcessor.from_pretrained(MODEL_ID)
model = ViTMAEForPreTraining.from_pretrained(MODEL_ID).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"Parameters: {n_params:,}")
print(f"Mask ratio: {model.config.mask_ratio}")

In [ ]:
def mae_reconstruct(model, processor, images, device):
    """Run MAE and convert logits to RGB images for display."""
    if not isinstance(images, list):
        images = [images]
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    # logits: (B, 196, 768) -> reshape to (B, 224, 224, 3)
    B, N, D = out.logits.shape
    p = model.config.patch_size
    h = w = model.config.image_size // p
    patches = out.logits.view(B, h, w, p, p, 3)
    recon = patches.permute(0, 1, 3, 2, 4, 5).reshape(B, 224, 224, 3)

    # Denormalize (ImageNet)
    mean = torch.tensor([0.485, 0.456, 0.406], device=recon.device)
    std = torch.tensor([0.229, 0.224, 0.225], device=recon.device)
    recon = recon * std + mean
    recon = recon.clamp(0, 1).cpu().numpy()
    return recon

---

## Section 4: Reconstructions Before Training

The model was pretrained on **ImageNet** (natural images). MRI scans look very different. Before we fine-tune, let's see how well it reconstructs MRI — likely mediocre, since the model has never seen medical images.

In [ ]:
# Reconstructions BEFORE fine-tuning
n_show = min(4, len(val_ds))
val_imgs = [to_rgb(val_ds[i][image_col]) for i in range(n_show)]
recon_before = mae_reconstruct(model, processor, val_imgs, device)

fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_before[i])
    axes[1, i].set_title("Reconstructed (before training)")
    axes[1, i].axis("off")
plt.suptitle("MAE Reconstructions — Before Fine-Tuning on MRI\n(Model pretrained on ImageNet only)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

---

## Section 5: Fine-Tune MAE on MRI

We use **HuggingFace Trainer** — minimal code, handles batching and optimization. The MAE model's forward pass returns `loss` when given `pixel_values`; Trainer uses that automatically.

- **Epochs:** 5 (fast on 300 images)
- **Batch size:** 16
- **Learning rate:** 1e-4

In [ ]:
def process_example(example):
    img = to_rgb(example[image_col])
    inputs = processor(images=img, return_tensors="pt")
    return {"pixel_values": inputs["pixel_values"].squeeze(0)}

train_ds.set_transform(process_example)
val_ds.set_transform(process_example)

# Verify one batch
ex = train_ds[0]
print(f"Example keys: {ex.keys()}")
print(f"pixel_values shape: {ex['pixel_values'].shape}")

In [ ]:
if not USE_PRECOMPUTED:
    training_args = TrainingArguments(
        output_dir="./vit_mae_mri",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        dataloader_num_workers=0,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    trainer.train()
    trainer.save_model("./vit_mae_mri")
    print("Model saved to ./vit_mae_mri")
else:
    model = ViTMAEForPreTraining.from_pretrained("./vit_mae_mri").to(device)
    print("Loaded fine-tuned model from ./vit_mae_mri")

In [ ]:
# Plot training and test loss
if not USE_PRECOMPUTED and trainer.state.log_history:
    train_steps = [e["step"] for e in trainer.state.log_history if "loss" in e]
    train_loss_vals = [e["loss"] for e in trainer.state.log_history if "loss" in e]
    eval_steps = [e["step"] for e in trainer.state.log_history if "eval_loss" in e]
    eval_loss_vals = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]

    plt.figure(figsize=(8, 4))
    plt.plot(train_steps, train_loss_vals, color="steelblue", linewidth=2, label="Train loss")
    if eval_steps:
        plt.plot(eval_steps, eval_loss_vals, color="coral", linewidth=2, marker="o",
                 markersize=5, label="Test loss")
    plt.xlabel("Step"); plt.ylabel("Loss")
    plt.title("MAE Fine-Tuning: Train and Test Loss on MRI")
    plt.legend()
    plt.tight_layout(); plt.show()

---

## Section 6: Reconstructions After Training

After fine-tuning on MRI images, the model has adapted to the medical imaging domain. Reconstructions should capture MRI-specific structures (anatomy, contrast) more accurately.

In [ ]:
# Reconstructions AFTER fine-tuning (same validation images)
model.eval()
recon_after = mae_reconstruct(model, processor, val_imgs, device)

fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_after[i])
    axes[1, i].set_title("Reconstructed (after training)")
    axes[1, i].axis("off")
plt.suptitle("MAE Reconstructions — After Fine-Tuning on MRI",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

---

## Section 7: Before vs After — Training Effect

Side-by-side comparison shows the improvement from domain adaptation. The model learns MRI-specific patterns (tissue contrast, anatomical structure) that it could not capture from ImageNet pretraining alone.

In [ ]:
# Before vs After: 3-row comparison
fig, axes = plt.subplots(3, n_show, figsize=(4*n_show, 10))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_before[i])
    axes[1, i].set_title("Before (ImageNet only)")
    axes[1, i].axis("off")
    axes[2, i].imshow(recon_after[i])
    axes[2, i].set_title("After (Fine-tuned on MRI)")
    axes[2, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Before Training", fontsize=12, fontweight="bold")
axes[2, 0].set_ylabel("After Training", fontsize=12, fontweight="bold")
plt.suptitle("Effect of MAE Fine-Tuning: Before vs After on MRI Domain",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("→ Fine-tuning on MRI images improves reconstruction quality. The model learns")
print("  domain-specific visual features that transfer to downstream medical tasks.")

---

## Section 8: Summary

| Step | What we did |
|------|-------------|
| 1 | Loaded **MRI-only** subset from ROCOv2 (80/20 train/val split) for fast iteration |
| 2 | Visualized **MAE masking** — 75% of patches hidden, encoder sees 25% |
| 3 | Used **HuggingFace** `ViTMAEForPreTraining` — no custom model code |
| 4 | Fine-tuned with **Trainer** — minimal training loop |
| 5 | Compared **before vs after** reconstructions to show domain adaptation |

### Key Takeaways

- **MAE** = self-supervised ViT pretraining by masking and reconstructing patches
- **High mask ratio (75%)** forces the model to learn semantic structure
- **Domain adaptation** — fine-tuning on MRI improves medical image representations
- **HuggingFace** — `ViTImageProcessor`, `ViTMAEForPreTraining`, `Trainer` handle the heavy lifting

### Next Steps

- Use the **encoder** (discard decoder) for downstream tasks: classification, segmentation
- Try larger MRI datasets for stronger adaptation
- Explore **MAE-v2**, **SparK** (sparse MAE) for extensions